# Synthetic Voice Classifier (ASVspoof 2019 LA)


In [ ]:
# Cell 1: Environment Setup & Drive Mount
!pip install -q transformers librosa torchaudio huggingface_hub scikit-learn
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/SyntheticVoiceDetector'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Working directory set to: {BASE_DIR}")


In [ ]:
# Cell 2: Dataset Acquisition (Kaggle Fallback)
import subprocess
from google.colab import userdata

os.chdir(BASE_DIR)

if not os.path.exists("LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"):
    print("Dataset not found. Downloading via Kaggle...")
    
    # Ensure kaggle is updated to a version that supports KAGGLE_API_TOKEN
    !pip install -U -q kaggle
    
    # Configure Kaggle API from Colab Secrets using single token
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    
    !kaggle datasets download -d anishsarkar22/asvpoof-2019-dataset-la -p . --unzip
    
    print("Dataset downloaded. Checking for nested archives...")
    import glob
    import zipfile
    import tarfile
    
    # Extract any nested zip files (often Kaggle mirrors have inner zips)
    for arc in glob.glob("**/*.zip", recursive=True):
        if "asvpoof-2019-dataset-la.zip" in arc: continue
        print(f"Extracting nested archive: {arc}")
        try:
            with zipfile.ZipFile(arc, 'r') as z:
                z.extractall(os.path.dirname(arc))
        except Exception as e:
            print(f"Failed to extract {arc}: {e}")
            
    # Extract any nested tar.gz files
    for arc in glob.glob("**/*.tar.gz", recursive=True) + glob.glob("**/*.tgz", recursive=True):
        print(f"Extracting nested archive: {arc}")
        try:
            with tarfile.open(arc, 'r:gz') as t:
                t.extractall(os.path.dirname(arc))
        except Exception as e:
            print(f"Failed to extract {arc}: {e}")
            
    print("Dataset download and extraction complete.")
else:
    print("Dataset already exists!")


In [ ]:
# Cell 3: Protocol Parsing
import pandas as pd

def load_protocol(file_path):
    # ASVspoof 2019 protocol format: SPEAKER_ID AUDIO_FILE_NAME NOT_USED SYSTEM_ID KEY
    df = pd.read_csv(file_path, sep=r"\s+", header=None, 
                     names=["speaker_id", "audio_file", "sys_id", "attack_type", "label"])
    return df

train_df = load_protocol("LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt")
dev_df = load_protocol("LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt")
eval_df = load_protocol("LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt")

print(f"Train samples: {len(train_df)}")
print(f"Dev samples: {len(dev_df)}")
print(f"Eval samples: {len(eval_df)}")


In [ ]:
# Cell 4: Feature Extraction Setup (Wav2Vec2 Frozen Backbone)
import torch
import librosa
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "facebook/wav2vec2-base"
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name).to(device)
model.eval()

def extract_embedding(audio_path):
    # Resample to 16kHz for wav2vec2
    audio, sr = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Mean-pool the last hidden state
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding


In [ ]:
# Cell 5: Diagnostic Check
print("Listing first 20 .flac files found in the dataset to verify extraction paths:")
!find . -iname "*.flac" | head -20

print("\nScanning directories (up to 3 levels deep):")
import os
for root, dirs, files in os.walk('.', topdown=True):
    # Limit depth to 3 levels
    depth = root.count(os.sep)
    if depth < 3:
        print(f"{root}/ ({len(dirs)} dirs, {len(files)} files)")



In [ ]:
# Cell 6: Extract and Cache Embeddings
import numpy as np
from tqdm.notebook import tqdm
import pickle
import glob

# Build a robust index mapping filename (without extension) to its full path across the entire dataset
print("Building global filename-to-path index for all audio files...")
# Match .flac and .wav just in case the mirror converted them
all_audio_files = glob.glob("**/*.flac", recursive=True) + glob.glob("**/*.wav", recursive=True)

# Map the base filename (e.g., 'LA_T_1138215') to the full path
filename_to_path = {os.path.splitext(os.path.basename(p))[0]: p for p in all_audio_files}
print(f"Indexed {len(filename_to_path)} audio files total.")

if len(filename_to_path) < 25000:
    print("WARNING: Found less than 25,000 audio files. The Kaggle download may have failed or timed out!")

def extract_features_for_split(df, split_name, cache_path):
    if os.path.exists(cache_path):
        print(f"Loading cached embeddings from {cache_path}")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)
            
    embeddings = []
    labels = []
    
    missing_count = 0
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        target_basename = str(row['audio_file'])
        file_path = filename_to_path.get(target_basename)
        
        if not file_path:
            missing_count += 1
            if missing_count == 1:
                print(f"Warning: Could not find first expected file '{target_basename}' (tried .flac and .wav).")
            continue
            
        emb = extract_embedding(file_path)
        embeddings.append(emb)
        labels.append(1 if row['label'] == 'spoof' else 0)
        
    if len(embeddings) == 0:
        raise FileNotFoundError(f"CRITICAL ERROR: No audio files were found for {split_name}! (Missing {missing_count} files)")
        
    X = np.array(embeddings)
    y = np.array(labels)
    
    with open(cache_path, 'wb') as f:
        pickle.dump((X, y), f)
        
    return X, y

print("Extracting Train features...")
X_train, y_train = extract_features_for_split(train_df, "Train", "X_train_v4.pkl")

print("Extracting Dev features...")
X_dev, y_dev = extract_features_for_split(dev_df, "Dev", "X_dev_v4.pkl")

print("Extracting Eval features...")
X_eval, y_eval = extract_features_for_split(eval_df, "Eval", "X_eval_v4.pkl")


In [ ]:
# Cell 7: Sanity Check and Class Balance
from sklearn.utils.class_weight import compute_class_weight

bonafide_count = np.sum(y_train == 0)
spoof_count = np.sum(y_train == 1)

print(f"Train Set - Bonafide: {bonafide_count}, Spoof: {spoof_count}")

if bonafide_count < 100 or spoof_count < 100:
    raise ValueError("CRITICAL ERROR: Severe class imbalance or data loading bug detected. Halting.")

# Compute weights
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = {0: weights[0], 1: weights[1]}
print(f"Computed Class Weights: {class_weights}")


In [ ]:
# Cell 8: Classifier Training
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("Training Logistic Regression classifier...")
clf = LogisticRegression(class_weight=class_weights, max_iter=1000)
clf.fit(X_train, y_train)

# Validate on Dev
y_dev_pred = clf.predict(X_dev)
dev_acc = accuracy_score(y_dev, y_dev_pred)
print(f"Dev Set Accuracy: {dev_acc:.4f}")


In [ ]:
# Cell 9: Held-Out Evaluation (Eval Set) & EER Computation
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve
import matplotlib.pyplot as plt

print("Evaluating on held-out EVAL partition (Contains unseen attacks A07-A19)...")

y_eval_pred = clf.predict(X_eval)
y_eval_proba = clf.predict_proba(X_eval)[:, 1] # Probability of being 'spoof'

acc = accuracy_score(y_eval, y_eval_pred)
prec = precision_score(y_eval, y_eval_pred)
rec = recall_score(y_eval, y_eval_pred)

print(f"Eval Accuracy:  {acc:.4f}")
print(f"Eval Precision: {prec:.4f}")
print(f"Eval Recall:    {rec:.4f}")

# EER Computation
fpr, tpr, thresholds = roc_curve(y_eval, y_eval_proba)
fnr = 1 - tpr

# Find the point where FPR and FNR are closest
diff = np.abs(fnr - fpr)
min_index = np.nanargmin(diff)
eer_threshold = thresholds[min_index]
eer = fpr[min_index]

print(f"\nEqual Error Rate (EER): {eer:.4f} (at threshold {eer_threshold:.4f})")

plt.plot(fpr, tpr, label='ROC Curve')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.scatter(fpr[min_index], tpr[min_index], color='red', marker='x', s=100, label=f'EER={eer:.4f}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.title('Receiver Operating Characteristic (EVAL Set)')
plt.show()


In [ ]:
# Cell 10: Export and HuggingFace Push
import joblib
from huggingface_hub import HfApi, create_repo

# Save locally to Drive
model_path = "synthetic_voice_classifier.joblib"
joblib.dump(clf, model_path)
print(f"Model saved to {model_path}")

# Push to HuggingFace Hub
hf_token = userdata.get('HF_TOKEN')
if hf_token:
    api = HfApi(token=hf_token)
    # Using a placeholder name, please change this if you run it
    repo_id = "your-username/synthetic-voice-detector-asvspoof" 
    
    try:
        create_repo(repo_id, private=True, repo_type="model", token=hf_token)
        print(f"Created new private repo: {repo_id}")
    except Exception as e:
        print(f"Repo creation response (might already exist): {e}")
        
    try:
        api.upload_file(
            path_or_fileobj=model_path,
            path_in_repo="synthetic_voice_classifier.joblib",
            repo_id=repo_id,
            repo_type="model",
            token=hf_token
        )
        print("Model successfully pushed to Hugging Face Hub!")
    except Exception as e:
        print(f"Upload failed: {e}")
else:
    print("HF_TOKEN not found in colab secrets, skipping push.")


In [ ]:
# Cell 11: Final Probe Test
print("Running final probe test on Eval subset...")

# Select a hard unseen spoof (e.g., A07 attack) and a real bonafide
bonafide_sample = eval_df[eval_df['label'] == 'bonafide'].iloc[0]
spoof_sample = eval_df[(eval_df['label'] == 'spoof') & (eval_df['attack_type'] == 'A07')].iloc[0]

# Use our global index to lookup the files safely (extension agnostic)
bonafide_path = filename_to_path.get(str(bonafide_sample['audio_file']))
spoof_path = filename_to_path.get(str(spoof_sample['audio_file']))

def probe(audio_path, true_label):
    emb = extract_embedding(audio_path).reshape(1, -1)
    pred_class = clf.predict(emb)[0]
    pred_proba = clf.predict_proba(emb)[0][1]
    
    print(f"File: {os.path.basename(audio_path)}")
    print(f"True Label: {true_label}")
    print(f"Prediction: {'Spoof' if pred_class == 1 else 'Bonafide'} (Spoof Prob: {pred_proba:.4f})\n")

probe(bonafide_path, "Bonafide")
probe(spoof_path, "Spoof (A07 Unseen Attack)")
